## Implementing transformer block from scratch


In [22]:
from torch import nn
import torch


class MultiHeadAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()

        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        # d_out lai num_heads le divide garna milnu parxa
        # each head ko equal dimension ko lagi

        self.d_out = d_out

        self.num_heads = num_heads

        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        # query projection layer banako

        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        # key projection layer banako

        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        # value projection layer banako

        self.out_proj = nn.Linear(d_out, d_out)
        # sabai heads combine pachi final projection garna use hunxa

        self.dropout = nn.Dropout(dropout)
        # dropout layer banako (overfitting reduce garna)

        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )
        # causal mask banako
        # future tokens block garna use hunxa

    def forward(self, x):

        b, num_tokens, d_in = x.shape
        # batch size, token count, input dimension nikaleko

        keys = self.W_key(x)
        # input lai key vectors ma transform gareko

        queries = self.W_query(x)
        # input lai query vectors ma transform gareko

        values = self.W_value(x)
        # input lai value vectors ma transform gareko

        # (b, num_tokens, d_out)
        # -> (b, num_tokens, num_heads, head_dim)

        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        # keys lai multiple heads ma split gareko

        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        # values lai multiple heads ma split gareko

        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        # queries lai multiple heads ma split gareko

        # (b, num_tokens, num_heads, head_dim)
        # -> (b, num_heads, num_tokens, head_dim)

        keys = keys.transpose(1, 2)
        # token ra head dimension swap gareko

        queries = queries.transpose(1, 2)
        # queries ko dimensions swap gareko

        values = values.transpose(1, 2)
        # values ko dimensions swap gareko

        attn_scores = queries @ keys.transpose(2, 3)
        # each head ko query ra key bich attention score nikaleko

        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        # required token size anusar mask select gareko

        attn_scores.masked_fill_(mask_bool, -torch.inf)
        # future token positions lai -inf banako
        # softmax pachi probability 0 hunxa

        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        # scaled softmax attention apply gareko

        attn_weights = self.dropout(attn_weights)
        # attention weights ma dropout apply gareko

        context_vec = (attn_weights @ values).transpose(1, 2)
        # weighted sum garera context vector banako
        # transpose garera original order ma lyako

        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        # sabai heads combine garera reshape gareko

        context_vec = self.out_proj(context_vec)
        # final linear projection apply gareko

        return context_vec
        # final multi-head attention output return gareko

In [23]:
import torch
import torch.nn as nn


# GELU activation function banako class
class GELU(nn.Module):

    # class initialize garna use huncha
    def __init__(self):
        # parent nn.Module ko constructor call gareko
        super().__init__()

    # forward pass define gareko
    def forward(self, x):

        # GELU formula apply gareko
        return (
            0.5  # formula ko constant value
            * x  # input value multiply gareko
            * (
                1
                + torch.tanh(  # tanh activation use gareko
                    # sqrt(2/pi) calculate gareko
                    torch.sqrt(torch.tensor(2.0 / torch.pi))
                    # x + 0.044715*x^3 calculate gareko
                    * (x + 0.044715 * torch.pow(x, 3))
                )
            )
        )

In [24]:
from torch import nn


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

In [25]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

## ya baat actual transformer ko code ho yo vanda mathi cahi previous build garako model use garako


In [26]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,  # Vocabulary size
    "context_length": 1024,  # Context length
    "emb_dim": 768,  # Embedding dimension
    "n_heads": 12,  # Number of attention heads
    "n_layers": 12,  # Number of layers
    "drop_rate": 0.1,  # Dropout rate
    "qkv_bias": False,  # Query-Key-Value bias
}

In [ ]:
from torch import nn


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"],
        )
        # multi-head attention layer banako

        self.ff = FeedForward(cfg)
        # feed forward network banako

        self.norm1 = LayerNorm(cfg["emb_dim"])
        # attention block agadi layer normalization

        self.norm2 = LayerNorm(cfg["emb_dim"])
        # feed forward block agadi layer normalization

        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])
        # shortcut connection ma dropout apply garna

    def forward(self, x):
        # Shortcut connection for attention block
        shortcut = x
        # original input lai shortcut ma save gareko

        x = self.norm1(x)
        # attention block agadi normalization apply gareko

        x = self.att(x)
        # multi-head attention apply gareko
        # output shape [batch_size, num_tokens, emb_size]

        x = self.drop_shortcut(x)
        # dropout apply gareko

        x = x + shortcut
        # residual connection: original input add gareko

        # Shortcut connection for feed forward block
        shortcut = x
        # attention output lai shortcut ma save gareko

        x = self.norm2(x)
        # feed forward block agadi normalization apply gareko

        x = self.ff(x)
        # feed forward network apply gareko

        x = self.drop_shortcut(x)
        # dropout apply gareko

        x = x + shortcut
        # residual connection: original input add gareko

        return x
        # final transformer block output return gareko

In [ ]:
import torch

torch.manual_seed(123)
# random number generator ko seed set gareko (reproducibility ko lagi)

x = torch.rand(2, 4, 768)  # A
# random input tensor banako
# shape: (batch_size=2, num_tokens=4, emb_dim=768)

block = TransformerBlock(GPT_CONFIG_124M)
# TransformerBlock initialize gareko (GPT config use garera)

output = block(x)
# input tensor lai transformer block ma pathaera output nikaleko

print("Input shape:", x.shape)
# input tensor ko shape print gareko

print("Output shape:", output.shape)
# output tensor ko shape print gareko

Input shape: torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768])
